# OptiCrop - 01 Data Analysis
This notebook performs Exploratory Data Analysis (EDA) on the crop recommendation dataset to understand feature distributions, correlations, outliers, and environmental patterns.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_PATH = Path("../data/Crop_recommendation.csv")
OUTPUT_DIR = Path("../screenshots/graphs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
EXPECTED_COLUMNS = FEATURE_COLUMNS + ["label"]

## A. Dataset Reading and Understanding

In [ ]:
df = pd.read_csv(DATA_PATH)
print("First five rows:")
display(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nDataset Information:")
df.info()
print("\nMissing values count per column:")
print(df.isnull().sum())
print("\nDuplicate rows count:", df.duplicated().sum())

## B. Univariate Analysis
Visualize feature distributions and the balance of the target label class.

In [ ]:
# Feature distributions
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for axis, column in zip(axes, FEATURE_COLUMNS):
    sns.histplot(df[column], bins=20, kde=True, ax=axis, color="#2d7a4a")
    axis.set_title(f"Distribution of {column}")
axes[-1].axis("off")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "univariate_distributions.png", dpi=200, bbox_inches="tight")
plt.show()

# Crop label counts
plt.figure(figsize=(12, 7))
sns.countplot(data=df, y="label", order=df["label"].value_counts().index, palette="viridis")
plt.title("Crop Label Distribution")
plt.xlabel("Count")
plt.ylabel("Crop")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "crop_label_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

## C. Bivariate Analysis
Explore interactions between specific environmental features and crops.

In [ ]:
# Humidity vs Rainfall scatterplot colored by crop
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x="humidity", y="rainfall", hue="label", alpha=0.7, palette="tab20")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=2)
plt.title("Humidity vs Rainfall Across Crop Records")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bivariate_humidity_rainfall.png", dpi=200, bbox_inches="tight")
plt.show()

# Temperature vs Crop boxplot
plt.figure(figsize=(15, 7))
sns.boxplot(data=df, x="label", y="temperature", palette="Set3")
plt.xticks(rotation=70)
plt.title("Temperature Variation Across Crops")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bivariate_temperature_by_crop.png", dpi=200, bbox_inches="tight")
plt.show()

# pH vs Crop boxplot
plt.figure(figsize=(15, 7))
sns.boxplot(data=df, x="label", y="ph", palette="Set2")
plt.xticks(rotation=70)
plt.title("Soil pH Variation Across Crops")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "bivariate_ph_by_crop.png", dpi=200, bbox_inches="tight")
plt.show()

## D. Multivariate Analysis

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(df[FEATURE_COLUMNS].corr(), annot=True, fmt=".2f", square=True, cmap="YlGnBu")
plt.title("Correlation Heatmap of Agricultural Features")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "multivariate_correlation_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()

# Multivariate pairplot of selected features
sample_df = df.sample(min(400, len(df)), random_state=42)
sns.pairplot(
    sample_df,
    vars=["N", "P", "K", "temperature", "humidity"],
    hue="label",
    corner=True,
    plot_kws={"alpha": 0.5, "s": 20}
)
plt.savefig(OUTPUT_DIR / "multivariate_pairplot.png", dpi=200, bbox_inches="tight")
plt.show()

## E. Outlier Analysis
Calculate Interquartile Range (IQR) for numeric variables. Outliers will be observed but not blindly deleted because extreme weather and soil levels represent realistic situations.

In [ ]:
# Boxplots of all features
plt.figure(figsize=(14, 6))
sns.boxplot(data=df[FEATURE_COLUMNS], palette="vlag")
plt.xticks(rotation=45)
plt.title("Outlier Inspection Using Boxplots")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "outlier_boxplots.png", dpi=200, bbox_inches="tight")
plt.show()

# Calculate IQR boundaries
for column in FEATURE_COLUMNS:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = ((df[column] < lower) | (df[column] > upper)).sum()
    print(f"{column}: lower={lower:.2f}, upper={upper:.2f}, potential outliers={outliers}")

## F. Seasonal Crop Analysis
Simple grouping of crops based on climate thresholds.

In [ ]:
summer = df[(df["temperature"] > 30) & (df["humidity"] > 50)]["label"].unique()
winter = df[(df["temperature"] < 20) & (df["humidity"] > 30)]["label"].unique()
rainy = df[(df["rainfall"] > 200) & (df["humidity"] > 50)]["label"].unique()

print("Summer crops:", sorted(summer))
print("Winter crops:", sorted(winter))
print("Rainy crops:", sorted(rainy))